# Data Preprocessing: BDD100K to COCO Format Conversion

This notebook handles the complete data preprocessing pipeline for the BDD100K dataset:

1. **Format Conversion**: Converts BDD100K annotations to COCO format
2. **Dataset Split**: Creates calibration (80%) and evaluation (20%) sets
3. **Validation**: Verifies data integrity using pycocotools

## Output Files:
- **val_calib.json** (80%): For Temperature Scaling calibration
- **val_eval.json** (20%): For final model evaluation

## Dataset Sources:
- https://www.kaggle.com/datasets/awsaf49/bdd100k-dataset
- https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k?resource=download ✓ (used)

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict
from PIL import Image
import numpy as np
from tqdm import tqdm
import shutil

HOME = Path(os.getcwd()).parent
BASE_DIR = HOME / "data"
BDD_DIR = BASE_DIR / "bdd100k"
COCO_DIR = BASE_DIR / "bdd100k_coco"

COCO_DIR.mkdir(exist_ok=True, parents=True)

print(f"✓ Home directory: {HOME}")
print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ BDD100K directory: {BDD_DIR}")
print(f"✓ COCO output directory: {COCO_DIR}")

## 1. Verify Dataset Structure

We verify that we have the annotations and validation images:

In [ ]:
labels_val = BDD_DIR / "labels" / "det_20" / "det_val.json"
images_val_dir = BDD_DIR / "images" / "100k" / "val"

if labels_val.exists():
    print(f"✓ Annotations found: {labels_val}")
    with open(labels_val) as f:
        annotations = json.load(f)
    print(f"  Total annotated images: {len(annotations)}")
else:
    print(f"✗ Annotations not found at: {labels_val}")

if images_val_dir.exists():
    image_files = list(images_val_dir.glob("*.jpg"))
    print(f"✓ Images directory: {images_val_dir}")
    print(f"  Total images: {len(image_files)}")
else:
    print(f"✗ Images directory not found: {images_val_dir}")

## 2. BDD100K → COCO Category Mapping

BDD100K has 10 object classes. We map them to COCO format with consecutive IDs:

In [ ]:
BDD_CATEGORIES = [
    "pedestrian",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motorcycle",
    "bicycle",
    "traffic light",
    "traffic sign"
]

categories_coco = []
category_name_to_id = {}

for idx, cat_name in enumerate(BDD_CATEGORIES, start=1):
    categories_coco.append({
        "id": idx,
        "name": cat_name,
        "supercategory": "object"
    })
    category_name_to_id[cat_name] = idx

print("COCO Categories:")
for cat in categories_coco:
    print(f"  ID {cat['id']}: {cat['name']}")

## 3. BDD100K → COCO Conversion Function

We convert BDD100K annotations (custom format) to standard COCO format:

In [ ]:
def convert_bdd_to_coco(bdd_annotations, images_dir, category_mapping):
    """
    Converts BDD100K annotations to COCO format.
    
    Args:
        bdd_annotations: List of annotations in BDD100K format
        images_dir: Directory containing the images
        category_mapping: Dictionary {category_name: coco_id}
    
    Returns:
        Dictionary in COCO format
    """
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": categories_coco
    }
    
    annotation_id = 1
    skipped_images = 0
    
    for img_idx, bdd_img in enumerate(tqdm(bdd_annotations, desc="Converting")):
        img_name = bdd_img["name"]
        img_path = images_dir / img_name
        
        if not img_path.exists():
            skipped_images += 1
            continue
        
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"Error opening image {img_name}: {e}")
            skipped_images += 1
            continue
        
        image_id = img_idx + 1
        coco_output["images"].append({
            "id": image_id,
            "file_name": img_name,
            "width": width,
            "height": height
        })
        
        if "labels" in bdd_img:
            for label in bdd_img["labels"]:
                category = label.get("category")
                
                if category not in category_mapping:
                    continue
                
                box2d = label.get("box2d")
                if not box2d:
                    continue
                
                x1 = box2d["x1"]
                y1 = box2d["y1"]
                x2 = box2d["x2"]
                y2 = box2d["y2"]
                
                bbox_width = x2 - x1
                bbox_height = y2 - y1
                
                if bbox_width <= 0 or bbox_height <= 0:
                    continue
                
                coco_output["annotations"].append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": category_mapping[category],
                    "bbox": [x1, y1, bbox_width, bbox_height],
                    "area": bbox_width * bbox_height,
                    "iscrowd": 0
                })
                annotation_id += 1
    
    print(f"\n✓ Images processed: {len(coco_output['images'])}")
    print(f"✓ Annotations created: {len(coco_output['annotations'])}")
    if skipped_images > 0:
        print(f"⚠ Images skipped: {skipped_images}")
    
    return coco_output

## 4. Convert Complete Dataset

We convert all BDD100K validation images to COCO format:

In [ ]:
with open(labels_val) as f:
    bdd_val_annotations = json.load(f)

print(f"Total images in BDD100K val: {len(bdd_val_annotations)}\n")

coco_val_full = convert_bdd_to_coco(
    bdd_val_annotations, 
    images_val_dir, 
    category_name_to_id
)

## 5. Split into Calibration and Evaluation

We split the validation set into:
- **val_calib.json** (80%): For Temperature Scaling calibration
- **val_eval.json** (20%): For final model evaluation

In [ ]:
np.random.seed(42)
split_ratio = 0.8

image_ids = [img["id"] for img in coco_val_full["images"]]
np.random.shuffle(image_ids)

split_idx = int(len(image_ids) * split_ratio)
calib_ids = set(image_ids[:split_idx])
eval_ids = set(image_ids[split_idx:])

print(f"Total images: {len(image_ids)}")
print(f"Calibration (80%): {len(calib_ids)} images")
print(f"Evaluation (20%): {len(eval_ids)} images")

def create_subset(coco_data, image_ids_subset):
    """Creates a COCO subset with specified images"""
    subset = {
        "images": [],
        "annotations": [],
        "categories": coco_data["categories"]
    }
    
    for img in coco_data["images"]:
        if img["id"] in image_ids_subset:
            subset["images"].append(img)
    
    for ann in coco_data["annotations"]:
        if ann["image_id"] in image_ids_subset:
            subset["annotations"].append(ann)
    
    return subset

coco_val_calib = create_subset(coco_val_full, calib_ids)
coco_val_eval = create_subset(coco_val_full, eval_ids)

print(f"\n✓ val_calib: {len(coco_val_calib['images'])} images, {len(coco_val_calib['annotations'])} annotations")
print(f"✓ val_eval: {len(coco_val_eval['images'])} images, {len(coco_val_eval['annotations'])} annotations")

## 6. Save COCO Files

We save the three JSON files in COCO format:

In [ ]:
output_files = {
    "val_full.json": coco_val_full,
    "val_calib.json": coco_val_calib,
    "val_eval.json": coco_val_eval
}

for filename, data in output_files.items():
    output_path = COCO_DIR / filename
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"✓ Saved: {output_path}")
    print(f"  - {len(data['images'])} images")
    print(f"  - {len(data['annotations'])} annotations")
    print()

## 7. Final Verification

We verify that files were created correctly and display statistics:

In [ ]:
def get_category_stats(coco_data):
    """Gets annotation statistics by category"""
    category_counts = defaultdict(int)
    for ann in coco_data["annotations"]:
        category_counts[ann["category_id"]] += 1
    return category_counts

print("=" * 60)
print("STATISTICS BY CATEGORY")
print("=" * 60)

for split_name, coco_data in [("val_calib", coco_val_calib), ("val_eval", coco_val_eval)]:
    print(f"\n{split_name.upper()}:")
    stats = get_category_stats(coco_data)
    for cat in categories_coco:
        count = stats.get(cat["id"], 0)
        print(f"  {cat['name']:20s}: {count:5d} annotations")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"✓ Dataset successfully converted to COCO format")
print(f"✓ Files saved to: {COCO_DIR}")
print(f"✓ val_calib: {len(coco_val_calib['images'])} images (80%)")
print(f"✓ val_eval: {len(coco_val_eval['images'])} images (20%)")
print(f"✓ Total categories: {len(categories_coco)}")
print("=" * 60)

## 📌 Next Steps

The generated COCO files are ready to be used:

1. **val_calib.json** → For training/calibrating Temperature Scaling (Phase 5)
2. **val_eval.json** → For final model evaluation with calibrated uncertainty

### Usage in Following Phases:

```python
# Phase 5: Load calibration data
calib_data = "data/bdd100k_coco/val_calib.json"

# Phase 5: Load evaluation data
eval_data = "data/bdd100k_coco/val_eval.json"
```

### Important Notes:

- ✓ Images remain in `data/bdd100k/images/100k/val/`
- ✓ JSON files only contain annotations and image references
- ✓ The split is reproducible (seed=42)
- ✓ No overlap between calibration and evaluation

In [ ]:
import os
from pathlib import Path

HOME = Path(os.getcwd()).parent
BDD100K_DIR = HOME / "data" / "bdd100k"
BDD100K_DIR.mkdir(exist_ok=True, parents=True)

print(f"Home: {HOME}")
print(f"BDD100K Directory: {BDD100K_DIR}")
print(f"Exists: {BDD100K_DIR.exists()}")

c:\Users\SP1VEVW\Desktop\projects\OVD-Model-ADAS\data
c:\Users\SP1VEVW\Desktop\projects\OVD-Model-ADAS\data\bdd100k ; exist: True


In [ ]:
# pip install pycocotools

  Using cached pycocotools-2.0.10-cp312-abi3-win_amd64.whl.metadata (1.3 kB)
Using cached pycocotools-2.0.10-cp312-abi3-win_amd64.whl (76 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime
import random
from pycocotools.coco import COCO

print("="*50)
print("1. VERIFYING DATA")
print("="*50)

VAL_IMAGES_DIR = BDD100K_DIR / "bdd100k" / "bdd100k" / "images" / "100k" / "val"
VAL_LABELS_FILE = BDD100K_DIR / "bdd100k_labels_release" / "bdd100k" / "labels" / "bdd100k_labels_images_val.json"

print(f"\nValidation Images: {VAL_IMAGES_DIR.exists()}")
print(f"Validation Labels: {VAL_LABELS_FILE.exists()}")

val_images = [f for f in os.listdir(VAL_IMAGES_DIR) if f.endswith('.jpg')] if VAL_IMAGES_DIR.exists() else []
print(f"Total validation images: {len(val_images)}")

if VAL_LABELS_FILE.exists():
    with open(VAL_LABELS_FILE, 'r') as f:
        val_labels = json.load(f)
    print(f"Total validation annotations: {len(val_labels)}")
else:
    print("Warning: Labels file not found")
    val_labels = []

categories_bdd = {}
for item in val_labels[:500]:
    for label in item.get('labels', []):
        cat = label.get('category')
        if cat:
            categories_bdd[cat] = categories_bdd.get(cat, 0) + 1

print(f"\nCategories found: {len(categories_bdd)}")
for cat, count in sorted(categories_bdd.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  - {cat}: {count}")

print("\n" + "="*50)
print("2. CONVERTING TO COCO FORMAT")
print("="*50)

CATEGORY_MAP = {
    'person': 1, 'rider': 2, 'car': 3, 'truck': 4, 'bus': 5,
    'train': 6, 'motorcycle': 7, 'bicycle': 8, 'traffic light': 9,
    'traffic sign': 10
}

CATEGORY_ALIASES = {
    'bike': 'bicycle'
}

def convert_to_coco(bdd_labels, images_dir, split_name):
    """Converts BDD100K to COCO format"""
    coco_format = {
        "info": {
            "description": f"BDD100K {split_name} Dataset - COCO Format",
            "version": "1.0",
            "year": 2024,
            "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }
    
    for cat_name, cat_id in CATEGORY_MAP.items():
        coco_format["categories"].append({
            "id": cat_id,
            "name": cat_name,
            "supercategory": "object"
        })
    
    annotation_id = 0
    
    for img_idx, item in enumerate(bdd_labels):
        img_name = item['name']
        img_path = images_dir / img_name
        
        if not img_path.exists():
            continue
        
        image_info = {
            "id": img_idx,
            "file_name": img_name,
            "width": 1280,
            "height": 720
        }
        coco_format["images"].append(image_info)
        
        for label in item.get('labels', []):
            category = label.get('category')
            
            if category in CATEGORY_ALIASES:
                category = CATEGORY_ALIASES[category]
            
            if category not in CATEGORY_MAP:
                continue
            
            box2d = label.get('box2d')
            if not box2d:
                continue
            
            x1 = box2d['x1']
            y1 = box2d['y1']
            x2 = box2d['x2']
            y2 = box2d['y2']
            
            width = x2 - x1
            height = y2 - y1
            area = width * height
            
            if width <= 0 or height <= 0:
                continue
            
            annotation = {
                "id": annotation_id,
                "image_id": img_idx,
                "category_id": CATEGORY_MAP[category],
                "bbox": [x1, y1, width, height],
                "area": area,
                "iscrowd": 0,
                "segmentation": []
            }
            coco_format["annotations"].append(annotation)
            annotation_id += 1
    
    return coco_format

print("\nConverting complete dataset...")
coco_val_full = convert_to_coco(val_labels, VAL_IMAGES_DIR, "validation_full")

print(f"Total images processed: {len(coco_val_full['images'])}")
print(f"Total annotations: {len(coco_val_full['annotations'])}")

print("\n" + "="*50)
print("3. SPLITTING INTO 80% TRAIN / 20% VAL")
print("="*50)

all_images = coco_val_full['images'].copy()
random.seed(42)
random.shuffle(all_images)

total_imgs = len(all_images)
train_size = int(total_imgs * 0.8)
val_size = total_imgs - train_size

train_images = all_images[:train_size]
val_images = all_images[train_size:]

print(f"\nTotal: {total_imgs} images")
print(f"Train: {train_size} images (80%)")
print(f"Val: {val_size} images (20%)")

train_img_ids = {img['id'] for img in train_images}
val_img_ids = {img['id'] for img in val_images}

def create_split(images, img_ids, split_name):
    split_data = {
        "info": coco_val_full["info"].copy(),
        "licenses": coco_val_full["licenses"],
        "images": images,
        "annotations": [],
        "categories": coco_val_full["categories"]
    }
    split_data["info"]["description"] = f"BDD100K {split_name} Dataset - COCO Format"
    
    for ann in coco_val_full['annotations']:
        if ann['image_id'] in img_ids:
            split_data['annotations'].append(ann)
    
    return split_data

train_coco = create_split(train_images, train_img_ids, "train")
val_coco = create_split(val_images, val_img_ids, "val")

print(f"\nTrain: {len(train_coco['images'])} imgs, {len(train_coco['annotations'])} anns")
print(f"Val: {len(val_coco['images'])} imgs, {len(val_coco['annotations'])} anns")

print("\n" + "="*50)
print("4. SAVING COCO FILES")
print("="*50)

COCO_OUTPUT_DIR = HOME / "data" / "bdd100k_coco"
COCO_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

train_file = COCO_OUTPUT_DIR / "val_calib.json"
val_file = COCO_OUTPUT_DIR / "val_eval.json"

with open(train_file, 'w') as f:
    json.dump(train_coco, f)
print(f"\n✓ Train saved: {train_file}")

with open(val_file, 'w') as f:
    json.dump(val_coco, f)
print(f"✓ Val saved: {val_file}")

print("\n" + "="*50)
print("5. FINAL STATISTICS")
print("="*50)

def print_stats(data, name):
    print(f"\n{name}:")
    print(f"  Images: {len(data['images'])}")
    print(f"  Annotations: {len(data['annotations'])}")
    if len(data['images']) > 0:
        print(f"  Average ann/img: {len(data['annotations'])/len(data['images']):.2f}")
    
    cat_dist = {}
    for ann in data['annotations']:
        cat_id = ann['category_id']
        cat_name = next(c['name'] for c in data['categories'] if c['id'] == cat_id)
        cat_dist[cat_name] = cat_dist.get(cat_name, 0) + 1
    
    print("  Top 5 categories:")
    for cat, count in sorted(cat_dist.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"    - {cat}: {count}")

print_stats(train_coco, "TRAIN")
print_stats(val_coco, "VAL")

print("\n" + "="*50)
print("✓ PROCESS COMPLETED")
print("="*50)
print(f"\nGenerated files:")
print(f"1. {train_file}")
print(f"2. {val_file}")
print(f"\nNow you can use COCOeval to calculate mAP, AP@50, F1, etc.")

print("\n" + "="*50)
print("6. VALIDATING WITH PYCOCOTOOLS")
print("="*50)

if train_file.exists():
    print("\n✓ Validating val_calib.json (80%)...")
    coco_calib = COCO(str(train_file))
    print(f"  - Loaded successfully")
    print(f"  - Images: {len(coco_calib.getImgIds())}")
    print(f"  - Categories: {len(coco_calib.getCatIds())}")
    print(f"  - Annotations: {len(coco_calib.getAnnIds())}")
    
if val_file.exists():
    print("\n✓ Validating val_eval.json (20%)...")
    coco_eval = COCO(str(val_file))
    print(f"  - Loaded successfully")
    print(f"  - Images: {len(coco_eval.getImgIds())}")
    print(f"  - Categories: {len(coco_eval.getCatIds())}")
    print(f"  - Annotations: {len(coco_eval.getAnnIds())}")
    
    print("\n✓ Available categories:")
    for cat in coco_calib.loadCats(coco_calib.getCatIds()):
        print(f"  - ID {cat['id']}: {cat['name']}")
    
    print("\n✓ COCO files validated correctly with pycocotools")
    print("  Ready to use with COCOeval to calculate metrics")


1. VERIFICANDO DATOS

Imágenes Val: True
Labels Val: True
Total imágenes Val: 10000
Total anotaciones Val: 10000

Categorías encontradas: 12
  - car: 5062
  - lane: 3808
  - traffic sign: 1754
  - traffic light: 1374
  - drivable area: 873
  - person: 746
  - truck: 212
  - bus: 91
  - bike: 40
  - rider: 35

2. CONVIRTIENDO A FORMATO COCO

Convirtiendo dataset completo...
Total anotaciones Val: 10000

Categorías encontradas: 12
  - car: 5062
  - lane: 3808
  - traffic sign: 1754
  - traffic light: 1374
  - drivable area: 873
  - person: 746
  - truck: 212
  - bus: 91
  - bike: 40
  - rider: 35

2. CONVIRTIENDO A FORMATO COCO

Convirtiendo dataset completo...
Total imágenes procesadas: 10000
Total anotaciones: 185074

3. DIVIDIENDO EN 80% TRAIN / 20% VAL

Total: 10000 imágenes
Train: 8000 imágenes (80%)
Val: 2000 imágenes (20%)

Train: 8000 imgs, 148515 anns
Val: 2000 imgs, 36559 anns

4. GUARDANDO ARCHIVOS COCO
Total imágenes procesadas: 10000
Total anotaciones: 185074

3. DIVIDIENDO 